In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# 컬렉션

In [ ]:
# docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

VECTOR_SIZE = 3072  # text-embedding-3-large 기준

qdrant = QdrantClient(host="localhost", port=6333)

In [3]:
qdrant.recreate_collection(
    collection_name="hts_case_all",
    vectors_config=qmodels.VectorParams(
        size=VECTOR_SIZE,
        distance=qmodels.Distance.COSINE,   # 벡터가 얼마나 비슷한 의미를 향하고 있는지 측정
    ),
)

print("Qdrant 컬렉션 생성 완료")

Qdrant 컬렉션 생성 완료


/var/folders/8v/lj7g5vl12t3c5ry4zkj9lkgh0000gn/T/ipykernel_1173/4051200919.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


In [4]:
# 문서 불러오기
import json

with open("/Users/minjikim/Desktop/sesac/project3/pretrait/cases_all(2021-2025).json", "r", encoding="utf-8") as f:
    cases = json.load(f)

In [5]:
len(cases)

4345

In [6]:
# 중복 데이터 있는지 확인
seen = []
duplicates = []

for item in cases:
    if item in seen:
        duplicates.append(item)
    else:
        seen.append(item)

print(f"전체 데이터 수: {len(cases)}")
print(f"완전히 동일한 중복 데이터 수: {len(duplicates)}")

for d in duplicates:
    print("중복 항목 발견:")
    print(d)

전체 데이터 수: 4345
완전히 동일한 중복 데이터 수: 0


In [8]:
import random

# 항상 같은 랜덤 결과 원하면 시드 고정 (재현성)
random.seed()

# test 랜덤 추출
test_cases = random.sample(cases, 145)

# train = 나머지
train_cases = [c for c in cases if c not in test_cases]

print("train 개수:", len(train_cases))  # 4200
print("test 개수:", len(test_cases))    # 145

train 개수: 4200
test 개수: 145


In [9]:
from langchain.schema import Document

# 청킹
docs = []

for case in train_cases:
    hts_code = case.get("결정세번", "")
    pdt_name = case.get("품명", "")
    pdt_detail = case.get("물품설명", "")
    pdt_reason = case.get("결정사유", "")

    # Document 생성
    doc = Document(
        page_content=pdt_detail,    # 임베딩에 사용할 텍스트 (이름만)
        metadata={
            "hts_code": hts_code,
            "pdt_name": pdt_name,
            "pdt_detail": pdt_detail,
            "pdt_reason": pdt_reason,
        }
    )
    docs.append(doc)

In [10]:
# 임베딩, 저장
from openai import OpenAI
from qdrant_client.http import models as qmodels
import uuid
import time

client = OpenAI(timeout=60)

BATCH_SIZE = 100   # ← 50~150 사이로 조절 가능

def embed_batch(texts, retries=3, delay=5):
    for attempt in range(1, retries + 1):
        try:
            res = client.embeddings.create(
                model="text-embedding-3-large",
                input=texts
            )
            return [d.embedding for d in res.data]

        except Exception as e:
            print(f"임베딩 실패 {attempt}회차: {e}")
            if attempt == retries:
                raise
            time.sleep(delay)

In [11]:
for i in range(0, len(docs), BATCH_SIZE):
    batch = docs[i:i + BATCH_SIZE]

    texts = [doc.page_content for doc in batch]

    # 배치 임베딩
    vectors = embed_batch(texts)

    points = []
    for doc, vec in zip(batch, vectors):
        points.append(
            qmodels.PointStruct(
                id=str(uuid.uuid4()),
                vector=vec,
                payload=doc.metadata
            )
        )

    # Qdrant 배치 업로드
    qdrant.upsert(
        collection_name="hts_case_all",
        points=points
    )

    print(f"업로드 완료: {i} ~ {i + len(batch) - 1}")

업로드 완료: 0 ~ 99
업로드 완료: 100 ~ 199
업로드 완료: 200 ~ 299
업로드 완료: 300 ~ 399
업로드 완료: 400 ~ 499
업로드 완료: 500 ~ 599
업로드 완료: 600 ~ 699
업로드 완료: 700 ~ 799
업로드 완료: 800 ~ 899
업로드 완료: 900 ~ 999
업로드 완료: 1000 ~ 1099
업로드 완료: 1100 ~ 1199
업로드 완료: 1200 ~ 1299
업로드 완료: 1300 ~ 1399
업로드 완료: 1400 ~ 1499
업로드 완료: 1500 ~ 1599
업로드 완료: 1600 ~ 1699
업로드 완료: 1700 ~ 1799
업로드 완료: 1800 ~ 1899
업로드 완료: 1900 ~ 1999
업로드 완료: 2000 ~ 2099
업로드 완료: 2100 ~ 2199
업로드 완료: 2200 ~ 2299
업로드 완료: 2300 ~ 2399
업로드 완료: 2400 ~ 2499
업로드 완료: 2500 ~ 2599
업로드 완료: 2600 ~ 2699
업로드 완료: 2700 ~ 2799
업로드 완료: 2800 ~ 2899
업로드 완료: 2900 ~ 2999
업로드 완료: 3000 ~ 3099
업로드 완료: 3100 ~ 3199
업로드 완료: 3200 ~ 3299
업로드 완료: 3300 ~ 3399
업로드 완료: 3400 ~ 3499
업로드 완료: 3500 ~ 3599
업로드 완료: 3600 ~ 3699
업로드 완료: 3700 ~ 3799
업로드 완료: 3800 ~ 3899
업로드 완료: 3900 ~ 3999
업로드 완료: 4000 ~ 4099
업로드 완료: 4100 ~ 4199


In [12]:
qdrant.get_collection("hts_case_all")

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=3400, points_count=4200, segments_count=3, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_s

# 평가

In [13]:
test_cases

[{'결정세번': '1901.90-2020',
  '품명': 'Food preparations of goods of headings 04.01 to 04.04; SLW 36004A; WHITE COMPOUND BUTTONS 25KG',
  '물품설명': 'Whey powder 15%, Skim milk powder 15%, Sugar 26%, Palm kernel oil, Lactose, Maltodextrin 등을 혼합·조제한 미백색의 원반 형상(지방분이 전 중량의 100분의 30 초과)    - 용도 : 제과용',
  '결정사유': 'ㅇ 관세율표 제1901호에는 “맥아 추출물(extract)과 고운 가루ㆍ부순 알곡ㆍ거친 가루ㆍ전분이나 맥아 추출물(extract)의 조제 식료품[코코아를 함유하지 않은 것이나 완전히 탈지(脫脂)한 상태에서 측정한 코코아의 함유량이 전 중량의 100분의 40 미만인 것으로 따로 분류되지 않은 것으로 한정한다], 제0401호부터 제0404호까지에 해당하는 물품의 조제 식료품[코코아를 함유하지 않은 것이나 완전히 탈지(脫脂)한 상태에서 측정한 코코아의 함유량이 전 중량의 100분의 5 미만인 것으로 따로 분류되지 않은 것으로 한정한다]”을 분류하며,    - 같은 호 HS해설서에서 “(Ⅲ) 제0401호부터 제0404호까지에 해당하는 물품의 조제 식료품 : 이 호의 조제품은 다음과 같은 점에서 제0401호부터 제0404호까지의 물품과 구별될 수 있다. 즉 그들은 천연밀크 구성성분 외에도, 제0401호부터 제0404호까지의 물품에는 허용되지 않는 그 밖의 성분을 함유하고 있다.”라고 설명하고 있음   ㅇ 또한, WCO 품목분류의견서를 수용한 관세청고시(제2001-62호, 2001.12.24.)에 따라 제0401호부터 제0404호까지에 해당하는 물품의 함량이 29% 이하인 제품에 대해서 제2106호에 분류하였음   ㅇ 따라서, 본 물품은 ‘제0401호부터 제0404호까지에 해당하는 물품(29% 초과)의 조제 식료품’으로서 지방분이 전 중

In [14]:
def embed(text):
    res = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    return res.data[0].embedding

In [15]:
import pandas as pd
import numpy as np

def evaluate_rag(test_cases, top_k=3, collection_name="hts_case_all"):
    """
    test_cases: [{'결정세번': ..., '품명': ..., '물품설명': ..., ...}, ...]
    top_k: RAG에서 몇 개까지 가져올지 (기본 3개)
    return: (df, accuracy)
      - df: 각 케이스별 통과/미통과 정보가 담긴 DataFrame
      - accuracy: 전체 통과 비율 (0~1)
    """
    records = []

    for i, case in enumerate(test_cases):
        # 1) 정답 코드 & 쿼리 텍스트 준비
        gt_code = str(case.get("결정세번", "")).strip()
        query_text = case.get("물품설명", "")  # 필요하면 품명도 같이 넣어도 됨

        # 2) 쿼리 임베딩
        query_vec = embed(query_text)

        # 3) RAG 검색
        results = qdrant.query_points(
            collection_name=collection_name,
            query=query_vec,
            limit=top_k,
        ).points

        # 4) 검색 결과에서 코드/스코어 뽑기
        retrieved_codes = [r.payload.get("hts_code") for r in results]
        retrieved_scores = [r.score for r in results]

        # 5) 통과 여부 (top_k 안에 정답 코드가 있으면 1, 아니면 0)
        is_pass = gt_code in retrieved_codes
        score = 1 if is_pass else 0

        records.append({
            "품명": case.get("품명", ""),
            "정답_결정세번": gt_code,
            "topk_hts_code": retrieved_codes,
            "topk_score": retrieved_scores,
            "pass": is_pass,
            "test_score": score,
        })

    df = pd.DataFrame(records)

    # 전체 accuracy (평균 점수)
    accuracy = df["test_score"].mean()
    retreived_accuracy = np.mean(df["topk_score"][0])

    return df, accuracy, retreived_accuracy

In [16]:
eval_df, acc, retreived_acc = evaluate_rag(test_cases, top_k=5)

print("전체 accuracy:", acc)    # 검색 정확도
print("유사도 점수: ", retreived_acc)  # 유사도 평균

전체 accuracy: 0.8413793103448276
유사도 점수:  0.76042929


유사도 점수가 0.75 이상인 결과값이 최종 프롬프트에서 1순위로 출력될 경우, 신뢰도 높음<br>
유사도 점수가 0.67 이상인 결과값이 최종 프롬프트에서 1순위로 출력될 경우, 신뢰도 중간<br>
유사도 점수가 0.55 이하인 결과값이 최종 프롬프트에서 1순위로 출력될 경우, 신뢰도 낮음<br>
> 유사도 점수 분포 확인 필요

In [47]:
# '결정세번': '1904.10-9000',
#  '품명': 'Prepared foods obtained by the swelling of cereal products; 귀리크런치',
#  '물품설명': '팽창된 밀, 팽창된 쌀, 볶은 귀리에 화이트 초콜릿(백설탕, 팜유, 탈지분유분말, 유청분말, 유지방, 글리세린지방산에스테르, 레시틴, 바닐린) 등을 혼합하여 성형한 바 형상의 것을 소포장한 후 수지제 봉지에 소매포장한 것(내용량 : 72g) [곡물의 표면에 화이트 초콜릿이 얇게 부분 도포된 형상임]   - 용도: 식용.',

query_text = "팽창된 밀, 팽창된 쌀, 볶은 귀리에 화이트 초콜릿(백설탕, 팜유, 탈지분유분말, 유청분말, 유지방, 글리세린지방산에스테르, 레시틴, 바닐린) 등을 혼합하여 성형한 바 형상의 것을 소포장한 후 수지제 봉지에 소매포장한 것(내용량 : 72g) [곡물의 표면에 화이트 초콜릿이 얇게 부분 도포된 형상임]   - 용도: 식용"
query_vec = embed(query_text)

# 3) RAG 검색
results = qdrant.query_points(
    collection_name="hts_case_all",
    query=query_vec,
    limit=5,
).points

print(results)

[ScoredPoint(id='04502682-a4e5-4a79-9e7f-ed3e67ede1c8', version=9, score=0.75007963, payload={'hts_code': '1904.10-9000', 'pdt_name': 'Prepared foods obtained by the swelling of cereal products; Konjac jjon-Deu-Ki', 'pdt_detail': 'ㅇ밀가루 39.2%, 현미 9%, 곤약분말 8.5%, 찰보리 8%, 옥수수가루 0.85%, 베이킹파우더 0.3%, 백설탕 29.4%, 물엿, D-소르비톨액, 정제소금, 자몽종자추출물을 혼합하여 압출성형공정을 통해 가열, 팽창시킨 길이 방향으로 결이 있는 미황색계 직사각형 모양의 곡물 팽창 제품을 수지제 팩에 포장한 것(내용량 : 35g)    - 용도: 식용', 'pdt_reason': 'ㅇ 관세율표 제1904호에는 "곡물이나 곡물 가공품을 팽창시키거나 볶아서 얻은 조제 식료품[예: 콘 플레이크(corn flake)]과 낟알 모양이나 플레이크(flake) 모양인 곡물(옥수수는 제외한다)과 그 밖의 가공한 곡물(고운 가루·부순 알곡·거친 가루는 제외하고 사전조리나 그 밖의 방법으로 조제한 것으로서 따로 분류되지 않은 것으로 한정한다)"이 분류되며, 소호 제1904.10호에는 "곡물이나 곡물가공품을 팽창시키거나 볶아서 얻은 조제 식료품"을 세분류하고 있음 . - 같은 호 해설서 제(A)항에서 ＂이 호에는 곡물(옥수수ㆍ밀ㆍ쌀ㆍ보리 등)을 팽창시키거나 볶아서 바삭바삭하게 만든 조제 식료품을 분류한다. 이는 우유를 가하거나 가하지 않고 주로 조반용 식료품으로 사용한다. 식염ㆍ설탕ㆍ당밀ㆍ맥아 추출물(malt extract)ㆍ과실이나 코코아(이 류의 주 제3호와 해설서 총설 참조)를 제조공정 중이나 후에 첨가할 때도 있다. 또한 이 군에는 고운 가루나 기울(bran)을 팽창시키거나 볶아서 얻은 유사한 식료품도 분류한다＂라고 설명하고 있음   ㅇ 따라서, 본 물품은 상

In [60]:
eval_df[eval_df["pass"] == False]

,품명,정답_결정세번,topk_hts_code,topk_score,pass,test_score
3,Food preparation; DD ESL YEAST RAISED MIX CONC...,2106.90-9099,"[1701.91-0000, 2106.90-9030, 2102.30-0000, 210...","[0.72274315, 0.7144853, 0.7106041, 0.6978068, ...",False,0
8,"Coffee powder, roasted and not decaffeinated; ...",0901.21-0000,"[2101.12-9090, 2101.11-1000, 1212.92-0000, 180...","[0.64032936, 0.6306628, 0.6183963, 0.616669, 0...",False,0
11,Non-alcoholic beverages; SOYBEAN MILK PLAIN,2202.99,"[2106.90-9099, 2106.90-9099, 1901.90-2010, 210...","[0.8049426, 0.7758621, 0.77253616, 0.7685399, ...",False,0
15,"Pumpkin, frozen ; FROZEN STEAMED PUMPKIN",0710.80-9000,"[0710.80-9090, 0710.40-0000, 0910.11-1000, 200...","[0.86790293, 0.7457961, 0.6670581, 0.65994585,...",False,0
17,Food preparations of goods of headings 04.01 t...,1901.90-2010,"[0406.10-1020, 0406.10-1020, 2106.90-9050, 040...","[0.69495565, 0.687662, 0.6739649, 0.6699252, 0...",False,0
18,Food preparation; EASY VIN CHAUD KIT,2106.90-9099,"[0811.90-9000, 0811.90-9000, 2008.99-9000, 200...","[0.70838326, 0.69564426, 0.69293374, 0.6918741...",False,0
24,Apple puree peparation; APPLE PASTE,2008.99-2000,"[2008.97-9000, 2008.97-9000, 2005.51-2000, 210...","[0.7345326, 0.7226385, 0.7002165, 0.69957244, ...",False,0
25,"Palm olein, refined; PALM OLEIN; RBD PALM OLEI...",1511.90-1000,"[1522.00-0000, 2106.90-9099, 1522.00-0000, 151...","[0.6444727, 0.6262446, 0.6134293, 0.60956585, ...",False,0
26,Chestnut preparation; VARIEGATO MARRONG GLACE,2008.19-1000,"[1702.90-1000, 1704.90-2090, 2106.90-2000, 210...","[0.6390211, 0.6181735, 0.616684, 0.6140871, 0....",False,0
28,Food preparation; SMOOTHIE KING DRICK POWDER M...,2106.10-9020,"[2106.90-9099, 2106.90-9099, 0404.90-2000, 190...","[0.7297787, 0.72331345, 0.6920521, 0.6777252, ...",False,0


# 최종 프롬프트

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


# 소호 번호까지 예측해주는 모델
def predict_hts(product_detail, context, rule, temperature, model):
    system_prompt = """
#역할과 목표
당신은 수출하는 물건의 HS코드를 결정해주는 능력있는 관세사입니다.
목표는 product_detail을 읽고 한국 HS코드 기준 10자리를 예측하는 것입니다.

#출력 규칙
다음 기준을 지켜주세요:
1. **product_detail의 내용물, 함량, 제조방법 충분히 고려**해야 합니다.
2. **rule과 context를 참고**하여 결과를 도출해야 합니다.
3. rule과 context의 사실과 위배되는 내용을 출력해서는 안됩니다.
4. **rule의 hs_code에 존재하지 않는 결정세번을 출력해서는 안됩니다.**
5. 분석 후, 가장 타당한 HS코드 순서대로 1순위, 2순위, 3순위를 배치하세요.


#출력 형식
1순위
결정세번: {{결정세번}} {{법령을 근거로 한 해당 결정세번 짧은 설명}} ex) 2202.10-9000
결정사유: {{결정사유}}
해당 결정세번이 결정된 이유를 법령과 유사 사례에 근거하여 설명한다. 만약 우려되는 부분이 있다면 추가 설명을 덧붙인다.

2순위(선택사항, 1순위, 2순위, 3순위 결정세번 중 같은 게 있을 경우 겹치는 결정세번은 생략하고 1순위 또는 2순위만 출력)
결정세번: {{결정세번}} {{법령을 근거로 한 해당 결정세번 짧은 설명}} ex) 2202.10-9000
결정사유: {{결정사유}}
해당 결정세번이 결정된 이유를 법령과 유사 사례에 근거하여 설명한다. 또 1순위, 3순위 결정세번과 분류 기준이 어떻게 다른지에 초점을 맞춰 설명한다.
만약 우려되는 부분이 있다면 추가 설명을 덧붙인다.

3순위(선택사항, 1순위, 2순위, 3순위 결정세번 중 같은 게 있을 경우 겹치는 결정세번은 생략하고 1순위 또는 2순위만 출력)
결정세번: {{결정세번}} {{법령을 근거로 한 해당 결정세번 짧은 설명}} ex) 2202.10-9000
결정사유: {{결정사유}}
해당 결정세번이 결정된 이유를 법령과 유사 사례에 근거하여 설명한다. 또 1순위, 2순위 결정세번과 분류 기준이 어떻게다른지에 초점을 맞춰 설명한다.
만약 우려되는 부분이 있다면 추가 설명을 덧붙인다.

#1순위, 2순위, 3순위 결정세번 중 같은 게 있을 경우
겹치는 결정세번은 생략하고 1개 또는 2개만 출력

#입력값 및 참고자료
아래 상품의 HS 소호를 대답해주세요.
---상품---
{product}

유사물품분류사례 (사례는 유사도 내림차순): 유사품목 품목명보다는, 품목의 설명이 유사한 것에 초점을 맞춰 분석
---context---
{context}

HS코드 분류기준 법령
---법령---
{rule}
법령에서 "기타"는 **같은 계층 다른 HS코드 중 그 어느 것에도 해당되지 않는 것**을 "기타"로 분류
""" 
    
    user_prompt = f"""
아래의 사용자 질의에 작성된 증상에 따라 가능성 있는 원인과 해결방법을 작성해줘.

--사용자 질의--
{product_detail}
    """

    prompt = ChatPromptTemplate.from_template([
        ("system", system_prompt),
        ("user", user_prompt)
    ])

    # 모델 정의 (gpt-, temperature=0)
    llm = ChatOpenAI(model=model, temperature=temperature)

    # chain생성
    chain = prompt | llm | StrOutputParser() 

    # 응답 출력 
    response = chain.invoke({
        "product": product_detail,
        "context": context,
        "rule": rule
    })

    return response 

In [58]:
product_detail = "팽창된 밀, 팽창된 쌀, 볶은 귀리에 화이트 초콜릿(백설탕, 팜유, 탈지분유분말, 유청분말, 유지방, 글리세린지방산에스테르, 레시틴, 바닐린) 등을 혼합하여 성형한 바 형상의 것을 소포장한 후 수지제 봉지에 소매포장한 것(내용량 : 72g) [곡물의 표면에 화이트 초콜릿이 얇게 부분 도포된 형상임] - 용도: 식용"

context = results

with open("/Users/minjikim/Desktop/sesac/project3/pretrait/hs_tree_251120.json", "r", encoding="utf-8") as f:
    rule = json.load(f)

print(predict_hts(product_detail, context, rule, 0, "gpt-5-mini-2025-08-07"))

1순위
결정세번: 1904.10-9000 관세율표 제1904호(곡물이나 곡물가공품을 팽창시키거나 볶아서 얻은 조제 식료품, 기타)
결정사유: 본 품목은 팽창된 밀·팽창된 쌀·볶은 귀리 등 곡물을 주성분으로 혼합·성형한 바 형태의 조제 식료품이고, 백색(화이트) 초콜릿은 표면에 얇게 부분 도포된 상태임. 관세율표 제1904호 및 HS 해설서(A항)는 “곡물(옥수수·밀·쌀·보리 등)을 팽창시키거나 볶아서 바삭바삭하게 만든 조제 식료품”을 제1904호에 분류하도록 규정하고 있으며(관세율표의 통칙 제1호·제6호 관련), 제공된 유사 판례들(분쇄·팽창한 곡물에 화이트초콜릿·다크초콜릿을 일부 코팅한 소포장 제품을 1904.10-9000으로 분류한 사례들)과 내용상 일치함. 또한 제19류 주 제3호 및 관련 판례에 따라 코코아(고형분) 함량이 완전히 탈지한 상태에서 총중량의 6%를 초과하거나 제품이 초콜릿으로 완전히 입혀져 초콜릿이 본질적 성분인 경우에는 제1806호로 제외되나, 본 물품은 표면에 얇게 부분 도포된 형태이고 구성성분(곡물류)이 지배적이므로 통상적으로 제1904.10-9000에 해당한다. 우려사항: 화이트초콜릿에 포함된 코코아(고형분) 함량 또는 제품의 초콜릿 피복 정도(전면 코팅인지, 함침·다량 도포인지)에 따라 제1806호 적용 가능성이 있으므로 수입 시 성분분석(코코아 고형분 비율) 또는 실물검사로 확인 권고.

2순위
결정세번: 1806.32-1000 제1806호 중 속을 채우지 않은 초콜릿류(초콜릿·코코아 함유 조제품)
결정사유: 만약 본 제품에서 “초콜릿(코코아를 함유한 조제품)”이 실질적으로 본질적 특성을 부여하거나(예: 초콜릿 코팅이 두껍고 전체를 완전히 입혔거나, 코코아 고형분이 완전히 탈지한 상태에서 측정하여 총중량의 6%를 초과하는 경우) 제품의 주된 성질이 ‘초콜릿 제품’으로 판단된다면 관세율표 제18류(제1806호)의 해당 항으로 분류된다. 1순위(제1904호)와의 차이는 본질적 성분(주된 원재료·주된 소비용도)이 곡물의 팽창 제품인지